In [1]:
!pip install -q transformers sentence-transformers python-docx faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving Psycology Complete Course.docx to Psycology Complete Course.docx


In [3]:
import os
os.rename(list(uploaded.keys())[0], "document.docx")

In [6]:
# This is what gave you those results:
from docx import Document

def load_docx(path):
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

# Standard chunking (400 chars with 50 overlap)
text = load_docx("/content/document.docx")
chunks = chunk_text(text)

print(f"Chunks: {len(chunks)}")

Chunks: 214


In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, convert_to_numpy=True)

print(embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(214, 384)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(question, k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    sims = cosine_similarity(q_emb, embeddings)[0]
    top_idx = sims.argsort()[-k:][::-1]
    return [chunks[i] for i in top_idx]

context_chunks = retrieve("What are the Roots of Psychology?", k=3)
combined_context = "\n\n".join(context_chunks)

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

def load_universal_model(model_name="google/flan-t5-base"):
    """Load model with universal settings for any document type"""
    print(f"Loading {model_name}...")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="left",
        truncation_side="left"
    )

    # Load model
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,  # Use float32 for CPU compatibility
        device_map="auto" if torch.cuda.is_available() else None
    )

    model.eval()
    print(f"✓ Model loaded successfully")
    return tokenizer, model

In [20]:
class UniversalQASystem:
    def __init__(self, model_name="google/flan-t5-base"):
        """Initialize a simple, working QA system"""
        print(f"Loading Universal QA System with {model_name}...")

        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        import torch

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,  # Use float32 for CPU
            device_map="auto" if torch.cuda.is_available() else None
        )
        self.model.eval()

        print(f"✓ Model loaded on {next(self.model.parameters()).device}")

    def answer(self, context, question, verbose=False):
        """Simple, reliable answer generation"""
        import re

        # Clean inputs
        clean_context = re.sub(r'\s+', ' ', context).strip()
        clean_question = re.sub(r'\s+', ' ', question).strip()

        if verbose:
            print(f"\n{'='*60}")
            print(f"QUESTION: {clean_question}")
            print(f"Context length: {len(clean_context)} characters")
            print(f"{'-'*60}")

        # SIMPLE prompt - this is what actually worked in your tests
        prompt = f"""Based on the following context, answer this question.

CONTEXT:
{clean_context[:1500]}

QUESTION: {clean_question}

ANSWER:"""

        try:
            # Tokenize
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
                padding=True
            ).to(self.model.device)

            # Generate with simple parameters
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=200,
                    num_beams=4,
                    temperature=0.5,
                    do_sample=False,
                    repetition_penalty=1.5,
                    length_penalty=0.8,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )

            # Decode
            answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Extract answer from response
            if "ANSWER:" in answer:
                answer = answer.split("ANSWER:")[-1].strip()

            # Clean up
            answer = re.sub(r'\s+', ' ', answer).strip()

            if verbose:
                print(f"ANSWER ({len(answer)} chars):")
                print(f"{'-'*40}")
                print(answer)
                print(f"{'-'*40}")
                if len(answer) > 50:
                    print("✓ Answer has substantial content")
                else:
                    print("⚠️  Answer might be too brief")

            return answer

        except Exception as e:
            error_msg = f"Error: {str(e)}"
            if verbose:
                print(error_msg)
            return error_msg

# Initialize the system
print("\n" + "="*80)
print("UNIVERSAL QA SYSTEM INITIALIZATION")
print("="*80)

universal_qa = UniversalQASystem()


UNIVERSAL QA SYSTEM INITIALIZATION
Loading Universal QA System with google/flan-t5-base...
✓ Model loaded on cpu


In [21]:
def batch_test(questions):
    """
    Test multiple questions at once
    """
    print(f"\n{'='*80}")
    print(f"BATCH TESTING - {len(questions)} QUESTIONS")
    print(f"{'='*80}")

    results = []
    for i, question in enumerate(questions, 1):
        print(f"\n[{i}/{len(questions)}] Question: {question}")

        # Retrieve context
        context_chunks = retrieve(question, k=3)
        combined_context = "\n\n".join(context_chunks)

        # Get answer (without verbose)
        answer = universal_qa.answer(combined_context, question, verbose=False)

        results.append({
            'question': question,
            'answer': answer,
            'context_length': len(combined_context),
            'answer_length': len(answer)
        })

        # Show summary
        print(f"   Answer: {answer[:100]}..." if len(answer) > 100 else f"   Answer: {answer}")
        print(f"   Context: {len(combined_context)} chars, Answer: {len(answer)} chars")

    # Print summary
    print(f"\n{'='*80}")
    print("TEST SUMMARY")
    print(f"{'='*80}")
    for result in results:
        print(f"Q: {result['question'][:50]}...")
        print(f"A: {result['answer'][:50]}... ({result['answer_length']} chars)")
        print()

    return results

# Example batch test
test_questions = [
    "What is psychology?",
    "Who is Wilhelm Wundt?",
    "What are psychological perspectives?",
    "Explain classical conditioning",
    "What is Maslow's hierarchy of needs?"
]

# Run batch test
results = batch_test(test_questions)


BATCH TESTING - 5 QUESTIONS

[1/5] Question: What is psychology?
   Answer: Psychology is the scientific study of behavior and mental processes
   Context: 904 chars, Answer: 67 chars

[2/5] Question: Who is Wilhelm Wundt?
   Answer: Established first psychology lab
   Context: 904 chars, Answer: 32 chars

[3/5] Question: What are psychological perspectives?
   Answer: Behavioral perspective: Stress is learned through conditioning.
   Context: 904 chars, Answer: 63 chars

[4/5] Question: Explain classical conditioning
   Answer: Pavlov’s dogs learned to salivate (CR) when hearing a bell
   Context: 904 chars, Answer: 58 chars

[5/5] Question: What is Maslow's hierarchy of needs?
   Answer: Five levels
   Context: 904 chars, Answer: 11 chars

TEST SUMMARY
Q: What is psychology?...
A: Psychology is the scientific study of behavior and... (67 chars)

Q: Who is Wilhelm Wundt?...
A: Established first psychology lab... (32 chars)

Q: What are psychological perspectives?...
A: Behavioral per